# LeVLJEPA

### model loading

In [12]:
import timm
import torch
from safetensors.torch import load_file

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create the backbone
levljepa_model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=False,
    num_classes=0,
).to(device)

# Load LeVLJEPA checkpoint
vision_weights = load_file(
    "PATH"
)

missing, unexpected = levljepa_model.load_state_dict(
    {
        k[len("encoder."):]: v
        for k, v in vision_weights.items()
        if k.startswith("encoder.")
    },
    strict=False
)

print("Missing:", missing)
print("Unexpected:", unexpected)

levljepa_model.eval()

Missing: []
Unexpected: []


VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False

# INFONCE

In [11]:
#
from huggingface_hub import hf_hub_download

config_path = hf_hub_download(
    repo_id="lukaskuhndkfz/InfoNCE-ViT-B-DataComp-200k",
    filename="config.json"
)

print(config_path)

C:\Users\louai\.cache\huggingface\hub\models--lukaskuhndkfz--InfoNCE-ViT-B-DataComp-200k\snapshots\f98f6d64ae9bd1324c0c59f766d3d8f8bdc823a6\config.json


In [12]:
import json

with open(config_path, "r") as f:
    config = json.load(f)

print(config)

{'model_type': 'infonce', 'model_size': 'base', 'vision_encoder': {'architecture': 'vit_base_patch16_224', 'hidden_size': 768, 'num_layers': 12, 'num_heads': 12}, 'text_encoder': {'architecture': 'gpt2', 'hidden_size': 768, 'num_layers': 12, 'num_heads': 12, 'vocab_size': 50257}, 'embed_dim': 512, 'projector_width': 2048, 'projector_depth': 4, 'training': {'dataset': 'DataComp-large', 'total_steps': 200000, 'batch_size': 4096}}


### model loading

In [2]:
import timm

model_infonce = timm.create_model(
    "vit_base_patch16_224",
    pretrained=False,
    num_classes=0,
)

#print(type(model_infonce))
#print(model_infonce)

In [4]:
# checkpoint loading into model
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download

vision_weights = load_file(
    hf_hub_download(
        "lukaskuhndkfz/InfoNCE-ViT-B-DataComp-200k",
        "vision_encoder.safetensors"
    )
)

missing, unexpected = model_infonce.load_state_dict(
    {k[len("encoder."):]: v
     for k, v in vision_weights.items()
     if k.startswith("encoder.")},
    strict=False
)

print("Missing:", missing)
print("Unexpected:", unexpected)

Missing: []
Unexpected: []


# OpenCLIP

In [2]:
# %pip install open_clip_torch

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   --------------------------- ------------ 1.0/1.5 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 5.9 MB/s  0:00:00
   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   -------------------- ------------------- 1.3/2.6 MB 7.4 MB/s eta 0:00:01
   ---------------------------------------- 2.6/2.6 MB 6.2 MB/s  0:00:00

  Attempting uninstall: timm

    Found existing installation: timm 0.4.12

   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
    Uninstalling timm-0.4.12:
   ---------------------------------------- 0/2 [timm]
      Successfully uninstalled timm-0.4.12
   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
   --------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
net2brain 1.3.1 requires timm==0.4.12, but you have timm 1.0.28 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
import open_clip
model, preprocess, tokenizer = open_clip.create_model_and_transforms(
    "ViT-B-16",
    pretrained="datacomp_l_s1b_b8k"
)
print(type(model))
model = model.to(device)
model.eval()

<class 'open_clip.model.CLIP'>


CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-11): 12 x ResidualAttentionBlock(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((768,), eps=1e-05, elementwise_affine

In [4]:
#print(model.visual)

VisionTransformer(
  (conv1): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
  (patch_dropout): Identity()
  (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (transformer): Transformer(
    (resblocks): ModuleList(
      (0-11): 12 x ResidualAttentionBlock(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (ls_1): Identity()
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): Sequential(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
        (ls_2): Identity()
      )
    )
  )
  (ln_post): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)


In [27]:
import torch.nn as nn

class OpenCLIPFeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

openclip_wrapper = OpenCLIPFeatureExtractor(model)

# feature extractor

In [5]:
# dataset loading
from net2brain.utils.download_datasets import DatasetBonnerPNAS2017, DatasetAlgonauts_NSD_Shared
from pprint import pprint
import os

#paths_bonner = DatasetBonnerPNAS2017.load_dataset()
#pprint(paths_bonner)

paths_NSD_shared  = DatasetAlgonauts_NSD_Shared.load_dataset()
#pprint(paths_NSD_shared)
#paths_NSD_shared.keys()


stimuli_path = paths_NSD_shared["images"]

# count files
# num_images = len([f for f in os.listdir(stimuli_path) if f.endswith(('.jpg', '.png'))])

#print("Number of images:", num_images)

layer_1  → block 1  
...  
layer_11 → block 11   
layer_12 → block 12 → final LayerNorm 

In [19]:
from torchvision import transforms as T
from torchvision import transforms as trn
import torchextractor as tx
from net2brain.feature_extraction import FeatureExtractor

def my_preprocessor(image, model_name, device):
    """
    Args:
        image (Union[Image.Image, List[Image.Image]]): A PIL Image or a list of PIL Images.
        model_name (str): The name of the model, used to determine specific preprocessing if necessary.
        device (str): The device to which the tensor should be transferred ('cuda' for GPU, 'cpu' for CPU).

    Returns:
        Union[torch.Tensor, List[torch.Tensor]]: The preprocessed image(s) as PyTorch tensor(s).
    """

    transforms = trn.Compose([
        trn.Resize((224, 224)),
        trn.ToTensor(),
        trn.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    img_tensor = transforms(image).unsqueeze(0)
    if device == 'cuda':
        img_tensor = img_tensor.cuda()

    return img_tensor


#-------------
#OpenClip 
#-------------
def my_openclip_transformer_extractor(
    preprocessed_data,
    layers_to_extract,
    model,
):
    visual = model.model.visual
    features = {}

    with torch.no_grad():

        # Patch embedding + CLS + positional embedding + ln_pre
        x = visual._embeds(preprocessed_data)

        for i, block in enumerate(visual.transformer.resblocks):

            x = block(x)

            if i < len(visual.transformer.resblocks) - 1:
                # Intermediate layer: CLS directly after transformer block
                features[f"layer_{i+1}"] = x[:, 0, :]

            else:
                # Final layer: final LayerNorm -> CLS
                x = visual.ln_post(x)
                features[f"layer_{i+1}"] = x[:, 0, :]

    return features
    
#----------
# LeVLJEPA. 
#-----------

def my_levljepa_transformer_extractor(
        preprocessed_data,
        layers_to_extract,
        model
):

    vit = model.model
    features = {}

    with torch.no_grad():

        # Patch embedding
        x = vit.patch_embed(preprocessed_data)

        # Add CLS token
        cls_token = vit.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_token, x), dim=1)

        # Positional embedding
        x = x + vit.pos_embed
        x = vit.pos_drop(x)

        # Transformer blocks
        for i, block in enumerate(vit.blocks):

            x = block(x)

            if i < len(vit.blocks) - 1:
                # Intermediate layer:
                # use patch tokens, not CLS
                patch_tokens = x[:, 1:, :]
                features[f"layer_{i+1}"] = patch_tokens.mean(dim=1)

            else:
                # Final layer:
                # apply the ViT's final LayerNorm first
                x = vit.norm(x)

                # Use patch tokens for LeVLJEPA dense representation
                patch_tokens = x[:, 1:, :]
                features[f"layer_{i+1}"] = patch_tokens.mean(dim=1)

    return features

def my_infonce_transformer_extractor(
        preprocessed_data,
        layers_to_extract,
        model,
):

    vit = model.model
    features = {}

    with torch.no_grad():

        x = vit.patch_embed(preprocessed_data)

        cls = vit.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls, x), dim=1)

        x = vit.pos_drop(x + vit.pos_embed)

        # Transformer blocks
        for i, block in enumerate(vit.blocks):

            x = block(x)

            # Intermediate layers:
            # CLS directly after the transformer block
            if i < len(vit.blocks) - 1:
                features[f"layer_{i+1}"] = x[:, 0, :]

            # Final layer:
            # transformer block -> final LayerNorm -> CLS
            else:
                x = vit.norm(x)
                features[f"layer_{i+1}"] = x[:, 0, :]

    return features
def my_cleaner(features):
    return features

In [14]:
import torch.nn as nn

class InfoNCEFeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

infonce_wrapper = InfoNCEFeatureExtractor(model_infonce).cuda()
levljepa_wrapper = InfoNCEFeatureExtractor(levljepa_model)

In [11]:
#infonce
from net2brain.feature_extraction import FeatureExtractor
import torch
from PIL import Image
import os



fx = FeatureExtractor(
    model=infonce_wrapper,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    preprocessor=my_preprocessor,
    feature_cleaner=my_cleaner,
    extraction_function=my_infonce_transformer_extractor
)

fx.extract(
    data_path=stimuli_path,
    save_path="PATH",
    consolidate_per_layer=True 
)

Processing files: 100%|████████████████████████████████████████████████| 872/872 [00:32<00:00, 27.18it/s, subfiles=1/1]


Consolidating data per layer...


100%|████████████████████████████████████████████████████████████████████████████████| 872/872 [00:59<00:00, 14.74it/s]


In [28]:
#openclip
fx = FeatureExtractor(
    model=openclip_wrapper,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    preprocessor=my_preprocessor,
    feature_cleaner=my_cleaner,
    extraction_function=my_openclip_transformer_extractor
)

fx.extract(
    data_path=stimuli_path,
    save_path="PATH",
    consolidate_per_layer=True
)

Processing files: 100%|████████████████████████████████████████████████| 872/872 [00:30<00:00, 28.94it/s, subfiles=1/1]


Consolidating data per layer...


100%|████████████████████████████████████████████████████████████████████████████████| 872/872 [01:06<00:00, 13.16it/s]


In [17]:
#levljepa
from net2brain.feature_extraction import FeatureExtractor
import torch
fx = FeatureExtractor(
    model=levljepa_wrapper,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    preprocessor=my_preprocessor,
    feature_cleaner=my_cleaner,
    extraction_function=my_levljepa_transformer_extractor
)

fx.extract(
    data_path=stimuli_path,
    save_path="PATH",
    consolidate_per_layer=True 
)

Processing files: 100%|████████████████████████████████████████████████| 872/872 [00:31<00:00, 27.90it/s, subfiles=1/1]


Consolidating data per layer...


100%|████████████████████████████████████████████████████████████████████████████████| 872/872 [00:54<00:00, 15.94it/s]


In [28]:
#
import inspect

print(inspect.getsource(model.forward))

    def forward(self, x):
        x = self.forward_features(x)
        if self.head_dist is not None:
            x, x_dist = self.head(x[0]), self.head_dist(x[1])  # x must be a tuple
            if self.training and not torch.jit.is_scripting():
                # during inference, return the average of both classifier predictions
                return x, x_dist
            else:
                return (x + x_dist) / 2
        else:
            x = self.head(x)
        return x



In [21]:
from net2brain.feature_extraction import FeatureExtractor

fx = FeatureExtractor(
    model="ViT-L/14",
    netset="Clip",
    device="cpu"
)

print(fx.layers)

AttributeError: 'FeatureExtractor' object has no attribute 'layers'